# Ch1 부록 — LLM의 안을 들여다본다 (logprobs · temperature)

교재 1절에서 "LLM은 다음 토큰을 예측할 뿐"이라고 했습니다. 이 노트북은 그 말을 **눈으로** 확인합니다.

- **실험 1 — logprobs**: 모델이 다음에 올 토큰을 어떤 확률로 고르는지 직접 봅니다. 확신할 때와 흔들릴 때가 보입니다. 이게 Kalai의 *"자신 있는 추측"* 환각의 뿌리입니다.
- **실험 2 — temperature**: 같은 입력을 온도만 바꿔 여러 번 돌립니다. 왜 `classify_one`이 추출에 `temperature=0`을 쓰는지 드러납니다.

> 실행 전: 레포 루트의 `.env`에 `OPENROUTER_API_KEY`가 있어야 합니다. 셀을 위에서부터 차례로 실행하세요.

In [ ]:
import os, math
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(Path.cwd() / ".env")  # 레포 루트 .env 로드
KEY = os.environ.get("OPENROUTER_API_KEY")
if not KEY or KEY == "sk-or-...":
    raise RuntimeError("OPENROUTER_API_KEY가 필요합니다. Ch0에서 .env에 실제 키를 넣은 뒤 다시 실행하세요.")
BASE = "https://openrouter.ai/api/v1"
# logprobs는 OpenAI 계열에서 노출된다(추론형 모델은 미지원). 그래서 gpt-4o-mini로 관찰한다.
LOGPROBS_MODEL = "openai/gpt-4o-mini"
print("준비 완료")

## 실험 1 — 다음 토큰 확률 분포

`logprobs=True`로 호출하면 모델이 고른 토큰과, 후보 토큰들의 확률을 함께 돌려줍니다. 두 질문을 비교해 보세요. 하나는 모델이 확신하고, 하나는 흔들립니다.

In [ ]:
llm = ChatOpenAI(model=LOGPROBS_MODEL, base_url=BASE, api_key=KEY,
                 temperature=0, logprobs=True, top_logprobs=5, max_tokens=3)

questions = [
    "다음 문장을 한 단어로 이어줘. '오늘 서울 날씨는'",
    "이 거래의 분류는? 식비/교통/생활 중 한 단어: 'GS25 도시락 4900원'",
]
for q in questions:
    r = llm.invoke(q)
    top = r.response_metadata["logprobs"]["content"][0]   # 첫 토큰의 분포
    print(f"\nQ: {q}\n   선택 → {top['token']!r}")
    for alt in top["top_logprobs"][:5]:
        p = math.exp(alt["logprob"])
        print(f"   {alt['token']!r:10} {p:6.1%} {'#' * int(p * 24)}")

**관찰.** 분류 질문은 한 토큰에 확률이 거의 몰립니다(확신). 날씨 질문은 여러 후보로 퍼집니다(불확실).

중요한 건, 불확실해도 모델은 **언제나 가장 확률 높은 토큰을 골라 답을 냅니다**. "모르겠다"가 기본값이 아닙니다. 이진 채점 벤치마크가 그 추측을 보상하기 때문이라는 게 Kalai 외(2025)의 분석이고, 그래서 프로덕션 에이전트는 Tool로 실제 값을 확인합니다. 우리 영수증 합계 검증이 그 가장 작은 형태입니다.

## 실험 2 — temperature와 추출의 재현성

temperature는 확률 분포를 얼마나 평평하게 만들지를 정합니다. `0`이면 늘 최상위 토큰, 높을수록 아래 후보도 뽑힙니다. 같은 입력을 온도만 바꿔 세 번씩 돌려 봅니다.

> **2026 함정.** **추론형 모델**은 온도 손잡이가 약해졌습니다 — 특히 o 시리즈(o1·o3)는 OpenAI 직결 시 1 이외 값에 400 에러를 냅니다. 게다가 **OpenRouter는 모델이 지원하지 않는 파라미터를 조용히 떨어뜨려서**, 추론형 모델엔 `temperature=0`을 줘도 에러 없이 무시될 수 있습니다. 반대로 비추론 모델(`gpt-4o-mini`, `gemini-3.5-flash`)에선 0이 정상 적용되니, 결정적 추출엔 그쪽을 씁니다.

In [ ]:
prompt = ("영수증 '광화문국밥 / 순대국밥 9000원 x 3 / 합계'에서 "
          "총액을 숫자로 적고, 한 줄 코멘트를 덧붙여줘.")

for temp in [0.0, 1.3]:
    m = ChatOpenAI(model=LOGPROBS_MODEL, base_url=BASE, api_key=KEY,
                   temperature=temp, max_tokens=40)
    outs = [m.invoke(prompt).content.strip() for _ in range(3)]
    print(f"\ntemperature={temp} — 3회 출력")
    for o in outs:
        print("   ·", o.replace(chr(10), ' ')[:48])

**관찰.** 총액(27,000)은 온도와 무관하게 안정적입니다. 하지만 자유롭게 생성되는 코멘트는 온도가 오를수록 매번 달라집니다.

추출(분류·정규화)에서 우리가 원하는 건 **같은 영수증 → 같은 RecordV1**입니다. 그래서 `classify_one.py`는 `temperature=0`으로 호출합니다. 창의성이 필요한 글쓰기(브리프 초안 같은)에서나 온도를 올립니다.

> 정리: logprobs는 "모델의 확신이 정답을 보장하지는 않는다"를, temperature는 "추출은 0으로 고정한다"를 확인하게 해 줍니다. 둘 다 뒤 챕터의 검증·HITL 설계로 이어집니다.